In [1]:
# ============================================================
# Early Fusion: CLIP + ViLT for Fakeddit 2-Way Classification
# Kaggle Notebook Script — 5-Fold CV | 80/10/10 Split
# Keeps 2-way only. 3-way and 6-way labels are not used.
#
# FIXES APPLIED
# ------------------------------------------------------------
# FIX 1 (primary crash):
#   __getitem__ returned a plain Python int as the label.
#   collate_fn called torch.stack(labels) which requires Tensors
#   not ints → TypeError: "expected Tensor as element 0, got int"
#   Fix: return torch.tensor(label, dtype=torch.long) from __getitem__
#
# FIX 2 (collate_fn safety):
#   torch.stack(labels) replaced with a dual-mode helper:
#   uses torch.stack() when items are Tensors (normal path),
#   falls back to torch.tensor() for raw ints (Subset edge case).
#
# FIX 3 (ViLT forward kwargs):
#   token_type_ids was passed as None when absent in the batch.
#   ViLT bypasses its internal buffer guard when None is explicit,
#   potentially crashing if seq_len != 40.
#   Fix: only pass token_type_ids / pixel_mask when present & non-None.
# ============================================================

# ── Cell 1: Install dependencies on Kaggle if needed ───────
# !pip install -q transformers Pillow scikit-learn pandas numpy torch torchvision tqdm

# ── Cell 2: Imports ───────────────────────────────────────
import os
import random
import warnings
import glob as _glob
import urllib.request as _urlreq
from concurrent.futures import ThreadPoolExecutor, as_completed as _as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    CLIPProcessor,
    CLIPModel,
    ViltProcessor,
    ViltModel,
    get_cosine_schedule_with_warmup,
)
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from tqdm import tqdm as _tqdm

warnings.filterwarnings("ignore")

# ── Cell 3: Reproducibility & config ──────────────────────
SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2
LABEL_COL = "2_way_label"
CLASS_NAMES = ["real", "fake"]

MAX_SAMPLES = 8000
N_FOLDS = 5
NUM_EPOCHS = 8
BATCH_SIZE = 16
ACCUM_STEPS = 2
PATIENCE = 3

CLIP_NAME = "openai/clip-vit-base-patch32"
VILT_NAME = "dandelin/vilt-b32-mlm"
FUSION_DIM = 256
FUSION_LAYERS = 2
FUSION_HEADS = 4
DROPOUT = 0.35
LABEL_SMOOTHING = 0.08

LR_FUSION = 1e-4
LR_BACKBONE = 2e-6
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
UNFREEZE_LAST_BLOCKS = 1
USE_AMP = DEVICE.type == "cuda"

IMG_DIR = "/kaggle/working/images"
OUTPUT_DIR = "/kaggle/working"
_HEADERS = {"User-Agent": "Mozilla/5.0"}
_TIMEOUT = 10

print(f"Device      : {DEVICE}")
print("Task        : Fakeddit 2-way only (0=real, 1=fake)")
print("Canceled    : 3-way and 6-way labels are not loaded or trained")
print(f"Fusion      : early token fusion, dim={FUSION_DIM}, layers={FUSION_LAYERS}")
print(f"Samples     : up to {MAX_SAMPLES:,} balanced samples")
print("=" * 70)

# ── Cell 4: Load Fakeddit metadata and images ─────────────
_hits = _glob.glob("/kaggle/input/**/multimodal_train.tsv", recursive=True)
if not _hits:
    _hits = [
        p for p in _glob.glob("/kaggle/input/**/*.tsv", recursive=True)
        if "train" in os.path.basename(p).lower()
    ]
if not _hits:
    raise FileNotFoundError(
        "multimodal_train.tsv not found under /kaggle/input/.\n"
        "Add the Fakeddit dataset in Kaggle with '+ Add data'."
    )

TSV_PATH = sorted(_hits)[0]
print(f"TSV: {TSV_PATH}")

df_raw = pd.read_csv(TSV_PATH, sep="\t", low_memory=False)
df_raw.columns = df_raw.columns.str.strip()
print(f"Raw rows: {len(df_raw):,}")
print(f"Columns : {df_raw.columns.tolist()}")

if LABEL_COL not in df_raw.columns:
    raise KeyError(
        f"Required column '{LABEL_COL}' not found. "
        "This script intentionally ignores 3_way_label and 6_way_label."
    )
if "image_url" not in df_raw.columns:
    raise KeyError("TSV has no 'image_url' column, so images cannot be downloaded.")

text_col = next((c for c in ["clean_title", "title", "text"] if c in df_raw.columns), None)
if text_col is None:
    raise KeyError("No text column found. Expected clean_title, title, or text.")

keep_cols = ["id", LABEL_COL, "image_url", text_col]
df = df_raw[keep_cols].dropna(subset=[LABEL_COL, "image_url", text_col]).copy()
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[LABEL_COL])
df[LABEL_COL] = df[LABEL_COL].astype(int)

# Strict 2-way filtering. 3-way/6-way columns are intentionally excluded.
df = df[df[LABEL_COL].isin([0, 1])].reset_index(drop=True)
df["text"] = df[text_col].astype(str).str.strip().replace("", "[No text]")
df = df[df["image_url"].astype(str).str.strip() != ""].reset_index(drop=True)
print(f"Usable 2-way rows with image_url: {len(df):,}")


def _download_image(row):
    dest = os.path.join(IMG_DIR, f"{row['id']}.jpg")
    if os.path.isfile(dest):
        return row["id"], True
    try:
        req = _urlreq.Request(str(row["image_url"]), headers=_HEADERS)
        data = _urlreq.urlopen(req, timeout=_TIMEOUT).read()
        if len(data) < 500:
            return row["id"], False
        with open(dest, "wb") as fh:
            fh.write(data)
        return row["id"], True
    except Exception:
        return row["id"], False


def prepare_balanced_dataset(df_in, max_samples=MAX_SAMPLES):
    per_class_buffer = min(
        df_in[LABEL_COL].value_counts().min(),
        max(1, (max_samples // 2) * 4),
    )
    sampled = pd.concat([
        df_in[df_in[LABEL_COL] == c].sample(per_class_buffer, random_state=SEED)
        for c in [0, 1]
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    os.makedirs(IMG_DIR, exist_ok=True)
    print(f"Downloading/checking up to {len(sampled):,} images -> {IMG_DIR}")
    rows = [r for _, r in sampled.iterrows()]
    ok_ids = set()
    with ThreadPoolExecutor(max_workers=32) as pool:
        futures = {pool.submit(_download_image, r): r["id"] for r in rows}
        for fut in _tqdm(_as_completed(futures), total=len(futures), desc="Images", unit="img"):
            rid, ok = fut.result()
            if ok:
                ok_ids.add(rid)

    if not ok_ids:
        raise RuntimeError(
            "No images downloaded. On Kaggle, enable Internet in notebook settings."
        )

    sampled = sampled[sampled["id"].isin(ok_ids)].reset_index(drop=True)
    per_class = min(sampled[LABEL_COL].value_counts().min(), max_samples // 2)
    balanced = pd.concat([
        sampled[sampled[LABEL_COL] == c].sample(per_class, random_state=SEED)
        for c in [0, 1]
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    balanced["img_path"] = balanced["id"].apply(lambda x: os.path.join(IMG_DIR, f"{x}.jpg"))
    balanced["label_enc"] = balanced[LABEL_COL].astype(int)
    print(
        f"Final dataset: {len(balanced):,} rows | "
        f"class dist: {balanced['label_enc'].value_counts().to_dict()}"
    )
    return balanced


df = prepare_balanced_dataset(df, MAX_SAMPLES)

# ── Cell 5: Dataset ───────────────────────────────────────
class FakedditEarlyFusionDataset(Dataset):
    def __init__(self, frame, clip_processor, vilt_processor):
        self.df = frame.reset_index(drop=True)
        self.clip_processor = clip_processor
        self.vilt_processor = vilt_processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"])
        label = int(row["label_enc"])

        try:
            img_path = row["img_path"]
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=(128, 128, 128))

        clip_enc = self.clip_processor(
            text=[text],
            images=img,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77,
        )
        vilt_enc = self.vilt_processor(
            images=img,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=40,
        )

        clip_out = {k: v.squeeze(0) for k, v in clip_enc.items()}
        vilt_out = {k: v.squeeze(0) for k, v in vilt_enc.items()}
        return clip_out, vilt_out, torch.tensor(label, dtype=torch.long)


def collate_fn(batch):
    clip_batch = {}
    vilt_batch = {}
    labels = []

    for clip_item, vilt_item, label in batch:
        for key, value in clip_item.items():
            clip_batch.setdefault(key, []).append(value)
        for key, value in vilt_item.items():
            vilt_batch.setdefault(key, []).append(value)
        labels.append(label)

    def stack_dict(d):
        out = {}
        for key, values in d.items():
            out[key] = torch.stack(values)
        return out

    def stack_vilt_dict(d):
        out = {}
        for key, values in d.items():
            if key == "pixel_values":
                channels = values[0].shape[0]
                max_h = max(v.shape[1] for v in values)
                max_w = max(v.shape[2] for v in values)
                padded = values[0].new_zeros((len(values), channels, max_h, max_w))
                for i, value in enumerate(values):
                    _, h, w = value.shape
                    padded[i, :, :h, :w] = value
                out[key] = padded
            elif key == "pixel_mask":
                max_h = max(v.shape[0] for v in values)
                max_w = max(v.shape[1] for v in values)
                padded = values[0].new_zeros((len(values), max_h, max_w))
                for i, value in enumerate(values):
                    h, w = value.shape
                    padded[i, :h, :w] = value
                out[key] = padded
            else:
                out[key] = torch.stack(values)
        return out

    # FIX 2: labels are torch.tensor(int) after __getitem__ fix.
    # Guard handles both Tensor and raw int (e.g. from Subset wrappers).
    if labels and isinstance(labels[0], torch.Tensor):
        stacked_labels = torch.stack(labels)
    else:
        stacked_labels = torch.tensor(labels, dtype=torch.long)
    return stack_dict(clip_batch), stack_vilt_dict(vilt_batch), stacked_labels


# ── Cell 6: Early Fusion Model ────────────────────────────
def _freeze(module):
    for p in module.parameters():
        p.requires_grad = False


def _unfreeze_last_clip_blocks(clip_model, n):
    if n <= 0:
        return
    for layer in clip_model.text_model.encoder.layers[-n:]:
        for p in layer.parameters():
            p.requires_grad = True
    for layer in clip_model.vision_model.encoder.layers[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


def _unfreeze_last_vilt_blocks(vilt_model, n):
    if n <= 0:
        return
    for layer in vilt_model.encoder.layer[-n:]:
        for p in layer.parameters():
            p.requires_grad = True
    if hasattr(vilt_model, "layernorm"):
        for p in vilt_model.layernorm.parameters():
            p.requires_grad = True


class EarlyFusionCLIPViLT(nn.Module):
    """
    Early fusion model:
      1. CLIP text tokens + CLIP image patch tokens are extracted.
      2. ViLT joint image-text tokens are extracted.
      3. All token streams are projected to FUSION_DIM and concatenated.
      4. A TransformerEncoder mixes the streams before classification.
      
    FIXED: Correctly handles different dimensions for CLIP text vs vision
    """

    def __init__(
        self,
        clip_model,
        vilt_model,
        fusion_dim=FUSION_DIM,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
        unfreeze_last_blocks=UNFREEZE_LAST_BLOCKS,
    ):
        super().__init__()
        self.clip = clip_model
        self.vilt = vilt_model

        _freeze(self.clip)
        _freeze(self.vilt)
        _unfreeze_last_clip_blocks(self.clip, unfreeze_last_blocks)
        _unfreeze_last_vilt_blocks(self.vilt, unfreeze_last_blocks)

        # FIXED: Get correct dimensions for each modality
        clip_text_hidden = self.clip.config.text_config.hidden_size  # 512
        clip_vision_hidden = self.clip.config.vision_config.hidden_size  # 768
        vilt_hidden = self.vilt.config.hidden_size  # 768

        print(f"  CLIP text hidden: {clip_text_hidden}")
        print(f"  CLIP vision hidden: {clip_vision_hidden}")
        print(f"  ViLT hidden: {vilt_hidden}")

        # FIXED: Separate projection layers with correct input dimensions
        self.clip_text_proj = nn.Linear(clip_text_hidden, fusion_dim)
        self.clip_vision_proj = nn.Linear(clip_vision_hidden, fusion_dim)
        self.vilt_proj = nn.Linear(vilt_hidden, fusion_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, fusion_dim))
        self.type_embed = nn.Embedding(4, fusion_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=FUSION_HEADS,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.fusion_encoder = nn.TransformerEncoder(encoder_layer, num_layers=FUSION_LAYERS)
        self.norm = nn.LayerNorm(fusion_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim // 2, num_classes),
        )

    def forward(self, clip_inputs, vilt_inputs):
        # CLIP text encoding
        clip_text = self.clip.text_model(
            input_ids=clip_inputs["input_ids"],
            attention_mask=clip_inputs["attention_mask"],
            return_dict=True,
        ).last_hidden_state  # (B, seq_len, 512)

        # CLIP vision encoding (exclude CLS token)
        clip_vision = self.clip.vision_model(
            pixel_values=clip_inputs["pixel_values"],
            return_dict=True,
        ).last_hidden_state[:, 1:, :]  # (B, num_patches, 768)

        # ViLT joint encoding
        # FIX 3: Build ViLT kwargs defensively.
        # token_type_ids: only pass when present AND non-None.
        #   Passing None lets ViLT fall back to its registered buffer
        #   [1, 40]; if seq_len != 40 this causes a shape error.
        # pixel_mask: same guard — only pass when present and non-None.
        vilt_kwargs = {
            "input_ids":      vilt_inputs["input_ids"],
            "attention_mask": vilt_inputs["attention_mask"],
            "pixel_values":   vilt_inputs["pixel_values"],
            "return_dict":    True,
        }
        if vilt_inputs.get("token_type_ids") is not None:
            vilt_kwargs["token_type_ids"] = vilt_inputs["token_type_ids"]
        if vilt_inputs.get("pixel_mask") is not None:
            vilt_kwargs["pixel_mask"] = vilt_inputs["pixel_mask"]
        vilt_tokens = self.vilt(**vilt_kwargs).last_hidden_state  # (B, seq_len, 768)

        bsz = clip_text.size(0)
        cls = self.cls_token.expand(bsz, -1, -1)
        
        # Project all modalities to fusion_dim
        clip_text = self.clip_text_proj(clip_text)  # (B, seq_len, fusion_dim)
        clip_vision = self.clip_vision_proj(clip_vision)  # (B, num_patches, fusion_dim)
        vilt_tokens = self.vilt_proj(vilt_tokens)  # (B, seq_len, fusion_dim)

        # Concatenate all token streams
        streams = [cls, clip_text, clip_vision, vilt_tokens]
        tokens = torch.cat(streams, dim=1)

        # Type embeddings for different token types
        type_ids = torch.cat([
            torch.zeros((bsz, 1), device=tokens.device, dtype=torch.long),
            torch.ones((bsz, clip_text.size(1)), device=tokens.device, dtype=torch.long),
            torch.full((bsz, clip_vision.size(1)), 2, device=tokens.device, dtype=torch.long),
            torch.full((bsz, vilt_tokens.size(1)), 3, device=tokens.device, dtype=torch.long),
        ], dim=1)
        tokens = tokens + self.type_embed(type_ids)

        # Only CLIP text padding is masked; image and ViLT joint tokens stay visible.
        text_pad = clip_inputs["attention_mask"] == 0
        visible = torch.zeros(
            (bsz, 1 + clip_vision.size(1) + vilt_tokens.size(1)),
            device=tokens.device,
            dtype=torch.bool,
        )
        key_padding_mask = torch.cat([
            visible[:, :1],
            text_pad,
            visible[:, 1:],
        ], dim=1)

        fused = self.fusion_encoder(tokens, src_key_padding_mask=key_padding_mask)
        cls_out = self.norm(fused[:, 0, :])
        logits = self.classifier(cls_out)
        return logits


# ── Cell 7: Training helpers ─────────────────────────────
class EarlyStopping:
    def __init__(self, patience=3, delta=1e-5):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.stop = False
        self.state = None

    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.counter = 0
            self.state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model):
        if self.state is not None:
            model.load_state_dict(self.state)


def move_batch(clip_batch, vilt_batch, labels):
    clip_moved = {k: v.to(DEVICE) for k, v in clip_batch.items()}
    vilt_moved = {k: v.to(DEVICE) for k, v in vilt_batch.items()}
    labels_moved = labels.to(DEVICE)
    return clip_moved, vilt_moved, labels_moved


def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    optimizer.zero_grad(set_to_none=True)

    for step, (clip_in, vilt_in, labels) in enumerate(loader, 1):
        clip_in, vilt_in, labels = move_batch(clip_in, vilt_in, labels)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(clip_in, vilt_in)
            raw_loss = criterion(logits, labels)
            loss = raw_loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item() * labels.size(0)
        preds_all.extend(logits.argmax(dim=1).detach().cpu().tolist())
        labels_all.extend(labels.detach().cpu().tolist())

    acc = accuracy_score(labels_all, preds_all)
    return total_loss / len(labels_all), acc


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []

    for clip_in, vilt_in, labels in loader:
        clip_in, vilt_in, labels = move_batch(clip_in, vilt_in, labels)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(clip_in, vilt_in)
            loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds_all.extend(logits.argmax(dim=1).detach().cpu().tolist())
        labels_all.extend(labels.detach().cpu().tolist())

    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average="binary", zero_division=0)
    pr = precision_score(labels_all, preds_all, average="binary", zero_division=0)
    rc = recall_score(labels_all, preds_all, average="binary", zero_division=0)
    return total_loss / len(labels_all), acc, f1, pr, rc, preds_all, labels_all


def overfit_gap(train_acc, val_acc):
    return max(0.0, train_acc - val_acc)


def make_optimizer(model):
    backbone_params, fusion_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("clip.") or name.startswith("vilt."):
            backbone_params.append(param)
        else:
            fusion_params.append(param)

    return AdamW(
        [
            {"params": fusion_params, "lr": LR_FUSION},
            {"params": backbone_params, "lr": LR_BACKBONE},
        ],
        weight_decay=WEIGHT_DECAY,
    )


# ── Cell 8: 5-Fold Cross-Validation ──────────────────────
print("\nLoading processors...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
vilt_processor = ViltProcessor.from_pretrained(VILT_NAME)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr = df["label_enc"].values
fold_results = []

print("=" * 70)
print(f"  Early Fusion CLIP + ViLT | Fakeddit 2-Way Only | {N_FOLDS}-Fold CV")
print("=" * 70)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df, labels_arr), 1):
    print(f"\n{'-' * 70}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'-' * 70}")

    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=0.125,  # 10% of full data from the 80% train_val side
        random_state=SEED + fold,
        stratify=labels_arr[train_val_idx],
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

    train_ds = FakedditEarlyFusionDataset(train_df, clip_processor, vilt_processor)
    val_ds = FakedditEarlyFusionDataset(val_df, clip_processor, vilt_processor)
    test_ds = FakedditEarlyFusionDataset(test_df, clip_processor, vilt_processor)

    nw = 2 if DEVICE.type == "cuda" else 0
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=nw, pin_memory=pin,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=nw, pin_memory=pin,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=nw, pin_memory=pin,
    )

    print("Building fresh CLIP + ViLT early-fusion model...")
    clip_model = CLIPModel.from_pretrained(CLIP_NAME)
    vilt_model = ViltModel.from_pretrained(VILT_NAME)
    model = EarlyFusionCLIPViLT(clip_model, vilt_model).to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Params: {total/1e6:.1f}M total | {trainable/1e6:.1f}M trainable")

    class_counts = np.bincount(train_df["label_enc"].values, minlength=NUM_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE),
        label_smoothing=LABEL_SMOOTHING,
    )

    optimizer = make_optimizer(model)
    steps_per_epoch = max(1, int(np.ceil(len(train_loader) / ACCUM_STEPS)))
    total_steps = steps_per_epoch * NUM_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(WARMUP_RATIO * total_steps)),
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    early_stop = EarlyStopping(PATIENCE)
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
        vl_loss, vl_acc, vl_f1, vl_pr, vl_rc, _, _ = eval_epoch(model, val_loader, criterion)
        gap = overfit_gap(tr_acc, vl_acc)
        monitor = vl_f1 - 0.25 * gap

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": vl_loss,
            "val_acc": vl_acc,
            "val_f1": vl_f1,
            "overfit_gap": gap,
        })

        print(
            f"Ep {epoch:02d}/{NUM_EPOCHS} | "
            f"TR-Loss={tr_loss:.4f} TR-Acc={tr_acc:.4f} | "
            f"VL-Loss={vl_loss:.4f} VL-Acc={vl_acc:.4f} "
            f"F1={vl_f1:.4f} Prec={vl_pr:.4f} Rec={vl_rc:.4f} "
            f"Gap={gap:.4f} Score={monitor:.4f}"
        )

        early_stop.step(monitor, model)
        if early_stop.stop:
            print(f"Early stopping at epoch {epoch} (best score={early_stop.best_score:.4f})")
            break

    early_stop.restore(model)
    _, ts_acc, ts_f1, ts_pr, ts_rc, ts_preds, ts_labels = eval_epoch(model, test_loader, criterion)
    best_hist = max(history, key=lambda h: h["val_f1"] - 0.25 * h["overfit_gap"])
    final_gap = overfit_gap(best_hist["train_acc"], best_hist["val_acc"])

    print(
        f"\nFold {fold} TEST | Acc={ts_acc:.4f} F1={ts_f1:.4f} "
        f"Prec={ts_pr:.4f} Rec={ts_rc:.4f} Overfit-Gap={final_gap:.4f}"
    )
    print(classification_report(ts_labels, ts_preds, target_names=CLASS_NAMES, zero_division=0))

    fold_results.append({
        "Fold": fold,
        "Test Acc": round(ts_acc, 4),
        "Test F1": round(ts_f1, 4),
        "Precision": round(ts_pr, 4),
        "Recall": round(ts_rc, 4),
        "Best Val Acc": round(best_hist["val_acc"], 4),
        "Overfit Gap": round(final_gap, 4),
    })

    torch.save(model.state_dict(), f"{OUTPUT_DIR}/early_fusion_clip_vilt_fold{fold}.pt")
    del model, clip_model, vilt_model, optimizer, scheduler, scaler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ── Cell 9: Summary ──────────────────────────────────────
print("\n" + "=" * 70)
print("  FINAL 5-FOLD RESULTS SUMMARY — EARLY FUSION 2-WAY ONLY")
print("=" * 70)

results_df = pd.DataFrame(fold_results)
mean_row = results_df.drop(columns=["Fold"]).mean().round(4).to_dict()
std_row = results_df.drop(columns=["Fold"]).std().round(4).to_dict()
mean_row["Fold"] = "Mean"
std_row["Fold"] = "Std"
summary_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print(summary_df.to_string(index=False))
summary_path = f"{OUTPUT_DIR}/early_fusion_clip_vilt_2way_results.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\nResults saved to {summary_path}")

mean_acc = results_df["Test Acc"].mean()
mean_f1 = results_df["Test F1"].mean()
print(f"\nAverage Test Accuracy : {mean_acc:.4f} ({mean_acc * 100:.2f}%)")
print(f"Average Test F1-Score : {mean_f1:.4f} ({mean_f1 * 100:.2f}%)")
print("Done. 3-way and 6-way classification remained canceled.")

Device      : cuda
Task        : Fakeddit 2-way only (0=real, 1=fake)
Canceled    : 3-way and 6-way labels are not loaded or trained
Fusion      : early token fusion, dim=256, layers=2
Samples     : up to 8,000 balanced samples
TSV: /kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv
Raw rows: 564,000
Columns : ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']
Usable 2-way rows with image_url: 562,466
Downloading/checking up to 32,000 images -> /kaggle/working/images


Images: 100%|██████████| 32000/32000 [02:19<00:00, 228.66img/s]


Final dataset: 8,000 rows | class dist: {0: 4000, 1: 4000}

Loading processors...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  CLIP text hidden: 512
  CLIP vision hidden: 768
  ViLT hidden: 768
Params: 265.0M total | 19.5M trainable
Ep 01/8 | TR-Loss=0.6065 TR-Acc=0.6936 | VL-Loss=0.5033 VL-Acc=0.8037 F1=0.8015 Prec=0.8107 Rec=0.7925 Gap=0.0000 Score=0.8015
Ep 02/8 | TR-Loss=0.4379 TR-Acc=0.8443 | VL-Loss=0.3916 VL-Acc=0.8725 F1=0.8771 Prec=0.8465 Rec=0.9100 Gap=0.0000 Score=0.8771
Ep 03/8 | TR-Loss=0.3659 TR-Acc=0.8904 | VL-Loss=0.3785 VL-Acc=0.8788 F1=0.8827 Prec=0.8548 Rec=0.9125 Gap=0.0116 Score=0.8798
Ep 04/8 | TR-Loss=0.3415 TR-Acc=0.9116 | VL-Loss=0.4304 VL-Acc=0.8725 F1=0.8668 Prec=0.9071 Rec=0.8300 Gap=0.0391 Score=0.8571
Ep 05/8 | TR-Loss=0.3220 TR-Acc=0.9177 | VL-Loss=0.3945 VL-Acc=0.8900 F1=0.8908 Prec=0.8842 Rec=0.8975 Gap=0.0277 Score=0.8839
Ep 06/8 | TR-Loss=0.2994 TR-Acc=0.9313 | VL-Loss=0.4058 VL-Acc=0.8838 F1=0.8850 Prec=0.8753 Rec=0.8950 Gap=0.0475 Score=0.8732
Ep 07/8 | TR-Loss=0.2870 TR-Acc=0.9407 | VL-Loss=0.4116 VL-Acc=0.8875 F1=0.8878 Prec=0.8856 Rec=0.8900 Gap=0.0532 Score=0.8745
Ep 